In [ ]:
# required installs
!pip install pandas numpy scikit-learn transformers sentencepiece torch

In [ ]:
# constants
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LEN = 128
BATCH_SIZE = 64

# change below as needed
TEST_CSV_PATH = "./data/test.csv"
MODEL_WEIGHTS_PATH = "Group57_C_best.pt"
OUTPUT_CSV_PATH = "Group_57_C.csv"

In [ ]:
# imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def generate_predictions(test_csv_path, model_weights_path, output_filename):
    print(f"loading test data from {test_csv_path}...")
    test_df = pd.read_csv(test_csv_path)
    test_df["premise"] = test_df["premise"].astype(str).fillna("")
    test_df["hypothesis"] = test_df["hypothesis"].astype(str).fillna("")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    
    # load the model
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    
    # load the weights to cpu, assuming no gpu available on marking machines
    model.load_state_dict(torch.load(model_weights_path, map_location=torch.device("cpu")))
    model.eval()

    formatted_results = []
    
    print("running predictions...")
    with torch.no_grad():
        for i in range(len(test_df)):
            row = test_df.iloc[i]
            # encode the premise/hypothesis pair
            inputs = tokenizer(
                row["premise"], 
                row["hypothesis"], 
                return_tensors="pt", 
                truncation=True, 
                max_length=MAX_LEN, 
                padding="max_length"
            )
            
            outputs = model(**inputs)
            # get the pair's prediction
            pred = torch.argmax(outputs.logits, dim=1).item()
            formatted_results.append(pred)
            
    output_df = pd.DataFrame({"prediction": formatted_results})
    output_df.to_csv(output_filename, index=False)
    print(f"done - file saved as {output_filename}")

generate_predictions(TEST_CSV_PATH, MODEL_WEIGHTS_PATH, OUTPUT_CSV_PATH)